# SVM and Tuning

This notebook is intentionally self-contained. It does not import local utility files, so the statistical idea, simulation, Python functions, evaluation, plots, and exam translation are all visible in one place.

## What problem are we solving?

Support vector machines classify by finding a separating boundary with a margin, and kernels allow nonlinear boundaries.

**Highest-probability exam pattern:** Tune `C` and `gamma` for an RBF SVM using stratified cross-validation.

## Assumptions, intuition, and theory

`C` controls how strongly the model penalizes margin violations. Larger C can fit the training data more tightly. `gamma` controls how local the RBF kernel is. Larger gamma creates more flexible boundaries.

## Python method notes and documentation

`StandardScaler` matters because SVMs use distances, `SVC` fits the classifier, `C` and `gamma` are core tuning parameters, and `cross_val_score` estimates validation accuracy.

Documentation links:

- [NumPy random generator](https://numpy.org/doc/stable/reference/random/generator.html)
- [pandas DataFrame](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)
- [matplotlib pyplot](https://matplotlib.org/stable/api/pyplot_summary.html)
- [SVC](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html)
- [LinearSVC](https://scikit-learn.org/stable/modules/generated/sklearn.svm.LinearSVC.html)
- [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html)
- [make_pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.make_pipeline.html)
- [StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html)
- [StratifiedKFold](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html)
- [cross_val_score](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html)

## Fully self-contained worked example

Every code line below is commented so it is easy to scan under exam time pressure.

In [ ]:
import numpy as np  # Import numerical arrays and random-number tools.
import pandas as pd  # Import DataFrame tables for tuning results.
import matplotlib.pyplot as plt  # Import plotting tools for tuning curves.
from sklearn.datasets import make_moons  # Import nonlinear classification simulator.
from sklearn.metrics import accuracy_score  # Import classification accuracy.
from sklearn.model_selection import StratifiedKFold, cross_val_score  # Import stratified CV tools.
from sklearn.pipeline import make_pipeline  # Import pipeline construction.
from sklearn.preprocessing import StandardScaler  # Import standardization for SVM distance calculations.
from sklearn.svm import SVC  # Import support vector classifier.
RANDOM_SEED = 123  # Store the reproducibility seed.
np.random.seed(RANDOM_SEED)  # Fix legacy NumPy randomness.
plt.style.use('default')  # Use a predictable plotting style.


In [ ]:
X, y = make_moons(n_samples=200, noise=0.25, random_state=RANDOM_SEED)  # Simulate nonlinear classes suitable for an RBF SVM.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)  # Define stratified 5-fold CV.
rows = []  # Create a list for grid-search results.
for C in [0.1, 1.0, 10.0]:  # Loop over margin-penalty values.
    for gamma in [0.1, 1.0, 10.0]:  # Loop over RBF kernel width values.
        model = make_pipeline(StandardScaler(), SVC(kernel='rbf', C=C, gamma=gamma))  # Build a standardized RBF SVM pipeline.
        scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy')  # Estimate accuracy by CV.
        rows.append({'C': C, 'gamma': gamma, 'mean_accuracy': np.mean(scores)})  # Store the grid result.
results = pd.DataFrame(rows)  # Convert tuning rows to a DataFrame.
best_row = results.loc[results['mean_accuracy'].idxmax()]  # Select the best tuning row.
display(results)  # Display the full tuning table.
print(f"Best C={best_row['C']}, gamma={best_row['gamma']}, accuracy={best_row['mean_accuracy']:.3f}")  # Print selected parameters.
pivot = results.pivot(index='C', columns='gamma', values='mean_accuracy')  # Reshape the results into a heatmap table.
plt.figure(figsize=(5, 4))  # Create a figure for the heatmap.
plt.imshow(pivot.values, aspect='auto', origin='lower')  # Plot accuracy values as an image.
plt.xticks(range(len(pivot.columns)), pivot.columns)  # Label gamma values on the x-axis.
plt.yticks(range(len(pivot.index)), pivot.index)  # Label C values on the y-axis.
plt.xlabel('gamma')  # Label the kernel-width axis.
plt.ylabel('C')  # Label the penalty axis.
plt.title('SVM CV accuracy grid')  # Title the heatmap.
plt.colorbar(label='CV accuracy')  # Add a colorbar for accuracy.
plt.show()  # Render the heatmap.


## Exam-style problem prompt

A prompt asks you to tune an SVM classifier. Build a small grid over C and gamma, use stratified CV accuracy, and report the best parameter pair.

## Adaptable code pattern

```python
# Step 1: Standardize predictors.
# Step 2: Choose a kernel, usually linear or RBF.
# Step 3: Create a grid over C and gamma for RBF.
# Step 4: Estimate accuracy by stratified CV.
# Step 5: Select the highest CV accuracy.
# Step 6: Explain C as penalty strength and gamma as locality.
```

## What to remember for the exam

For SVM exams, always mention scaling. `C` and `gamma` are not cosmetic; they control the bias-variance tradeoff.